In [1]:
# Loading Packages
import pandas as pd
import numpy as np
import random
import os
import torch
import faiss
import json
import re
import shutil
import random
from tqdm import tqdm
from torch import Tensor
from dotenv import load_dotenv
from huggingface_hub import login
from torch.utils.data import DataLoader
from beir.datasets.data_loader import GenericDataLoader
from transformers import AutoTokenizer, logging, AutoModel, AutoModelForCausalLM
logging.set_verbosity_error()

/work/mbouthil/.conda/envs/myuwenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
### Loading data ###
data_dir = "/work/mbouthil/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

100%|██████████| 8841823/8841823 [01:18<00:00, 112654.19it/s]


In [12]:
random.seed(42)
J=20_000
qrels = dict(random.sample(list(qrels.items()), J))

In [4]:
system_prompt = '''You are a subject matter expert in your field with substantial accumulated knowledge in a
specific subject or topic, validated by academic degrees, certifications, and/or years of
professional experience in that field.

Write a question that elaborates on the provided passage(s). Ensure that your question is answered by the passage(s). 
Provide only the question and format it as follows:**question**. 
'''

In [5]:
# Loading LM
from packages.llama import Llama_LM
llama = Llama_LM()

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 583.24it/s, Materializing param=model.norm.weight]                              


In [15]:
def batch_splits(item:list, batch_size:int=64):

    for i in range(0, len(item), batch_size):
        yield item[i:i + batch_size]

batches = batch_splits(list(qrels.items()), batch_size=1)

In [18]:
syn_dict = dict()

for batch in batches:

    q_ids, p_data = zip(*batch)
    passages = []

    # Formatting passages for LLM message
    for entry in p_data:
        p_ids = list(entry.keys())
        p_ids = [key for key, value in entry.items() if value > 0]

        if len(p_ids) > 1:
            p_texts = []
            for i, p_id in enumerate(p_ids):
                p_texts.append(f"passage {i+1}: {corpus[p_id]['text']}")
            passages.append("\n".join(p_texts))

        else:
            passages.append(corpus[p_ids[0]]['text'])


    messages = [
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": passage}
        ]
        for passage in passages
    ]

    syn_queries = llama.prompt(messages)

    for i, q_id in enumerate(q_ids):
        syn_dict[q_id] = syn_queries[i]
    break

In [19]:
syn_dict

{'755300': 'What percentage of individuals with HoFH have a mutation in both alleles of the LDLR gene?'}